## Experiments #2 and #4: targeted training

Counterpart of the random control (#3 / #5, `exp3_exp5_random_control.ipynb`) with exactly the same configuration and method, shared through `utils/finbert_cnn.py`:

1. Baseline CNN on FinBERT (`yiyanghkust/finbert-pretrain`) word-piece embeddings, trained on the full training set.
2. A copy of the baseline is trained further on the full training set with the **hard subset up-weighted** (weight 1.5, Adam 1e-4, early stopping on val macro F1). The hard subset is the experiment 7 P20 training hard set.
3. Score on the validation set (overall / hard / non-hard, macro F1). Hard subsets are the experiment 7 P20 subsets, loaded from `results/hard_samples/`.

- **#2**: FinBERT embeddings, frozen (identical to the experiment 7 targeted run at P20)
- **#4**: FinBERT embeddings, fine-tuned (baseline and fine-tuning both with trainable embeddings)

The only difference with #3 / #5 is *which* sentences get up-weighted. Test set is not used.

In [1]:
import os
os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from tensorflow import keras
from tqdm import tqdm

from utils import finbert_cnn
from utils.finbert_cnn import BASELINE_SEED, SEEDS, load_finbert_inputs, load_hard_selection, train_baseline
from utils.preprocessing import encode_labels
from utils.random_control import (
    HARD_PERCENTILE, embedding_name, summarize_seeds, to_metrics_rows, update_metrics_csv,
)
from utils.targeted import CONDITION, run_targeted

# Same seed as experiment 7: the baseline comes out identical.
keras.utils.set_random_seed(BASELINE_SEED)

### Data (FinBERT word-piece tokenization, max length 64)

In [2]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)

X_train, X_val, embedding_matrix = load_finbert_inputs(train_df, val_df)
data = {"X_train": X_train, "y_train": y_train, "X_val": X_val, "y_val": y_val}
print(X_train.shape, X_val.shape, embedding_matrix.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(2901, 64) (967, 64) (30873, 768)


### Frozen baseline and hard subsets (experiment 7)
- **Training** hard set: bottom 20% margin of out-of-fold CNN predictions. These are the sentences that get up-weighted.
- **Validation** hard set: the training cutoff applied to the baseline's validation predictions.

Both are loaded from `results/hard_samples/` (saved by notebook 07), the same sets the random control uses.

In [3]:
baseline_frozen = train_baseline(embedding_matrix, X_train, y_train, X_val, y_val, trainable=False)

hard_train_idx, val_hard_idx = load_hard_selection(HARD_PERCENTILE)
print(f"train hard (up-weighted): {len(hard_train_idx)} / {len(y_train)}   |   val hard: {len(val_hard_idx)} / {len(y_val)}")

Epoch 1/20
90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5882 - loss: 0.9495 - val_macro_f1: 0.3184
91/91 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.5888 - loss: 0.9485 - val_accuracy: 0.6236 - val_loss: 0.8556 - val_macro_f1: 0.3184
Epoch 2/20
89/91 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6380 - loss: 0.8344 - val_macro_f1: 0.4151
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6412 - loss: 0.8309 - val_accuracy: 0.6784 - val_loss: 0.7745 - val_macro_f1: 0.4151
Epoch 3/20
89/91 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6854 - loss: 0.7536 - val_macro_f1: 0.4368
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6880 - loss: 0.7501 - val_accuracy: 0.6939 - val_loss: 0.7141 - val_macro_f1: 0.4368
Epoch 4/20
90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7205 - loss: 0.6522 - val_macro_f1: 0.5897
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7218 - loss: 0.6512 - val_accuracy: 0.7332 - val_loss: 0.6519 - val_macro_f1: 0.5897
Epoch 5/20
88

### Fine-tuned-embedding baseline (#4)

In [4]:
keras.utils.set_random_seed(BASELINE_SEED)
baseline_finetuned = train_baseline(embedding_matrix, X_train, y_train, X_val, y_val, trainable=True)

Epoch 1/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.6005 - loss: 0.9262 - val_macro_f1: 0.3910
91/91 ━━━━━━━━━━━━━━━━━━━━ 11s 103ms/step - accuracy: 0.6005 - loss: 0.9262 - val_accuracy: 0.6598 - val_loss: 0.8027 - val_macro_f1: 0.3910
Epoch 2/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.6998 - loss: 0.7193 - val_macro_f1: 0.5998
91/91 ━━━━━━━━━━━━━━━━━━━━ 8s 89ms/step - accuracy: 0.6998 - loss: 0.7193 - val_accuracy: 0.7446 - val_loss: 0.6443 - val_macro_f1: 0.5998
Epoch 3/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.8335 - loss: 0.4394 - val_macro_f1: 0.7122
91/91 ━━━━━━━━━━━━━━━━━━━━ 8s 85ms/step - accuracy: 0.8335 - loss: 0.4394 - val_accuracy: 0.7808 - val_loss: 0.5636 - val_macro_f1: 0.7122
Epoch 4/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.9338 - loss: 0.2080 - val_macro_f1: 0.7239
91/91 ━━━━━━━━━━━━━━━━━━━━ 7s 82ms/step - accuracy: 0.9338 - loss: 0.2080 - val_accuracy: 0.7859 - val_loss: 0.6220 - val_macro_f1: 0.7239
Epoch 5/20

### Run: #2 (frozen) and #4 (fine-tuned), all seeds

In [5]:
rows = []
for trainable, baseline in [(False, baseline_frozen), (True, baseline_finetuned)]:   # False = #2, True = #4
    for seed in tqdm(SEEDS, desc=embedding_name(trainable)):
        rows.append(run_targeted(baseline, data, trainable, seed, hard_train_idx, val_hard_idx))
seed_df = pd.DataFrame(rows)
seed_df

finbert_frozen:   0%|          | 0/5 [00:00<?, ?it/s]

 - val_macro_f1: 0.7385
 - val_macro_f1: 0.7472
 - val_macro_f1: 0.7358


finbert_frozen:  20%|██        | 1/5 [00:34<02:18, 34.68s/it]

 - val_macro_f1: 0.7414
 - val_macro_f1: 0.7348


finbert_frozen:  40%|████      | 2/5 [01:07<01:40, 33.61s/it]

...

 - val_macro_f1: 0.7011
 - val_macro_f1: 0.7185
 - val_macro_f1: 0.7064


finbert_finetuned:  80%|████████  | 4/5 [04:32<01:14, 74.24s/it]

 - val_macro_f1: 0.7119
 - val_macro_f1: 0.7113


finbert_finetuned: 100%|██████████| 5/5 [06:15<00:00, 75.02s/it]


,seed,embedding_config,n_extra,weight,overlap_with_hard,embedding_change,accuracy_overall,f1_overall,accuracy_hard,f1_hard,accuracy_nonhard,f1_nonhard,f1_negative,f1_neutral,f1_positive
0,42,finbert_frozen,580,1.5,1.0,0.000000,0.796277,0.747175,0.618557,0.444444,0.816092,0.769619,0.682692,0.852245,0.706587
1,43,finbert_frozen,580,1.5,1.0,0.000000,0.794209,0.741371,0.597938,0.418048,0.816092,0.768800,0.669951,0.851133,0.703030
2,44,finbert_frozen,580,1.5,1.0,0.000000,0.796277,0.747790,0.628866,0.456913,0.814943,0.767580,0.685714,0.852245,0.705411
3,45,finbert_frozen,580,1.5,1.0,0.000000,0.809721,0.766154,0.659794,0.491801,0.826437,0.779760,0.707547,0.861869,0.729045
4,46,finbert_frozen,580,1.5,1.0,0.000000,0.795243,0.746395,0.618557,0.444444,0.814943,0.769104,0.685990,0.851792,0.701403
5,42,finbert_finetuned,580,1.5,1.0,0.000282,0.775595,0.704974,0.525773,0.437774,0.803448,0.740564,0.597015,0.845843,0.672065
6,43,finbert_finetuned,580,1.5,1.0,0.000108,0.774560,0.705811,0.556701,0.464831,0.798851,0.737241,0.607843,0.844302,0.665289
7,44,finbert_finetuned,580,1.5,1.0,0.000106,0.780765,0.708448,0.567010,0.493503,0.804598,0.738144,0.587065,0.850407,0.687873
8,45,finbert_finetuned,580,1.5,1.0,0.000191,0.783868,0.718490,0.556701,0.494615,0.809195,0.748797,0.610837,0.848980,0.695652
9,46,finbert_finetuned,580,1.5,1.0,0.000107,0.778697,0.711923,0.556701,0.494615,0.803448,0.741138,0.604878,0.847097,0.683794


### Check: #2 reproduces the experiment 7 targeted run (P20)

In [6]:
exp7 = pd.read_csv("../results/experiment_7_percentile_seeds.csv")
exp7 = exp7[(exp7.percentile == HARD_PERCENTILE) & (exp7.strategy == "targeted")].astype({"seed": int})
ours = seed_df[seed_df.embedding_config == embedding_name(False)]
check = ours.merge(exp7, on="seed")[["seed", "f1_overall", "macro_f1", "f1_hard", "hard_macro_f1",
                                     "accuracy_overall", "accuracy", "accuracy_hard", "hard_accuracy"]]
pairs = [("f1_overall", "macro_f1"), ("f1_hard", "hard_macro_f1"),
         ("accuracy_overall", "accuracy"), ("accuracy_hard", "hard_accuracy")]
print("identical to experiment 7:", all(np.allclose(check[a], check[b]) for a, b in pairs))
check

identical to experiment 7: True


,seed,f1_overall,macro_f1,f1_hard,hard_macro_f1,accuracy_overall,accuracy,accuracy_hard,hard_accuracy
0,42,0.747175,0.747175,0.444444,0.444444,0.796277,0.796277,0.618557,0.618557
1,43,0.741371,0.741371,0.418048,0.418048,0.794209,0.794209,0.597938,0.597938
2,44,0.747790,0.747790,0.456913,0.456913,0.796277,0.796277,0.628866,0.628866
3,45,0.766154,0.766154,0.491801,0.491801,0.809721,0.809721,0.659794,0.659794
4,46,0.746395,0.746395,0.444444,0.444444,0.795243,0.795243,0.618557,0.618557


### Results (mean and standard deviation over seeds)

In [7]:
mean, std = summarize_seeds(seed_df)
display(mean.round(3))
display(std.round(3))

,accuracy_overall,f1_overall,accuracy_hard,f1_hard,accuracy_nonhard,f1_nonhard,f1_negative,f1_neutral,f1_positive
embedding_config,,,,,,,,,
finbert_finetuned,0.779,0.71,0.553,0.477,0.804,0.741,0.602,0.847,0.681
finbert_frozen,0.798,0.75,0.625,0.451,0.818,0.771,0.686,0.854,0.709


,accuracy_overall,f1_overall,accuracy_hard,f1_hard,accuracy_nonhard,f1_nonhard,f1_negative,f1_neutral,f1_positive
embedding_config,,,,,,,,,
finbert_finetuned,0.004,0.005,0.016,0.025,0.004,0.005,0.010,0.002,0.012
finbert_frozen,0.006,0.010,0.023,0.027,0.005,0.005,0.014,0.005,0.011


### Save (per-seed details + one averaged row per experiment in the shared metrics.csv)

In [8]:
seed_df.to_csv("../results/targeted_seeds.csv", index=False)
update_metrics_csv("../results/metrics.csv", to_metrics_rows(mean, CONDITION))

,condition,embedding_config,architecture,accuracy_overall,f1_overall,accuracy_hard,f1_hard,accuracy_nonhard,f1_nonhard,f1_negative,f1_neutral,f1_positive
0,baseline,tfidf_1-2gram,logreg,0.776629,0.717241,NaN,NaN,NaN,NaN,0.631579,0.845771,0.674374
1,baseline,random_init_100d,lstm_64_dropout,0.698035,0.464626,NaN,NaN,NaN,NaN,0.000000,0.832653,0.561224
2,baseline,tfidf_1-2gram,logreg,0.776629,0.717241,NaN,NaN,NaN,NaN,0.631579,0.845771,0.674374
3,baseline,random_init_100d,lstm_64_dropout,0.698035,0.464626,NaN,NaN,NaN,NaN,0.000000,0.832653,0.561224
4,random_control,finbert_finetuned,cnn_64_finbert,0.779731,0.710225,0.538776,0.466455,0.806904,0.743418,0.597771,0.847918,0.684985
5,random_control,finbert_frozen,cnn_64_finbert,0.798759,0.749278,0.618367,0.455547,0.819102,0.771809,0.683070,0.854425,0.710340
6,targeted,finbert_finetuned,cnn_64_finbert,0.778697,0.709929,0.552577,0.477068,0.803908,0.741177,0.601528,0.847326,0.680935
7,targeted,finbert_frozen,cnn_64_finbert,0.798345,0.749777,0.624742,0.451130,0.817701,0.770973,0.686379,0.853857,0.709095
